# 12 · End-to-end walkthrough (read-only)

This notebook walks one reviewer through the whole project, from the raw-data audit to the headline results, in one
scroll. It **trains nothing and creates no new result**: it calls the repository's existing code, loads saved
checkpoints for inference only, and reads committed result files. Every cell has a short note above it: what the cell
does and what to look at.

Inputs on Kaggle (Internet ON, CPU): notebook 04's output (dataset `dvdhere/results-train-04`: cleaned splits, frozen
scaler, run `sweep_f1.0_s42`) and notebook 06's output (kernel `dvdhere/training-06`: the FU2 relabel model). The code is
cloned from GitHub. The only files written are the figure set in `docs/figures/v2/` (PNG + the plotted numbers as CSV).

## 1 · Setup

**What it does:** clones the repository, imports `common`, `s2_model`, `s3_continual` and `s5_explain`, and finds the two
attached inputs with the repository's own marker search (`s5_explain.find_dir`), never a hard-coded path.
**Look at:** the code commit (every number below comes from this commit's code and committed files) and the two input
folders.

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time, copy
from pathlib import Path

START_CWD = os.getcwd()
REPO = "https://github.com/DevChi1l/cicddos2019-continual-nids"
CODE = Path(os.environ.get("NIDS_CODE", "/kaggle/working/code"))
if not (CODE / "s3_continual.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(CODE)], check=True)
sys.path[:0] = [str(CODE), str(CODE / "dashboard")]
os.chdir(CODE)
COMMIT = subprocess.run(["git", "-C", str(CODE), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

import common
import s2_model
import s3_continual as S3
import s5_explain as S5
import s1b_experiences as S1B
import build_dashboard as BD   # the dashboard's loaders: same committed files, same numbers as the dashboard

SRC = S5.find_dir("runs/cii/sweep_f1.0_s42/config.json", "NIDS_SRC")                  # notebook 04 output (nids_out)
FU2 = S5.find_dir("fu2/cii/fu2_relabel_f1.0_s42/poisoned_relabel/A2.pt", "NIDS_FU2")  # notebook 06 output (nids_out)
assert SRC is not None, "attach dvdhere/results-train-04 (notebook 04's output)"
assert FU2 is not None, "attach the output of kernel dvdhere/training-06 (FU2)"
RES = CODE / "experiments" / "results"           # committed result files
RUN = SRC / "runs" / "cii" / "sweep_f1.0_s42"    # every stored Syn label flipped to BENIGN (f = 1.0), seed 42
FU2RUN = FU2 / "fu2" / "cii" / "fu2_relabel_f1.0_s42"
FIG = Path(os.environ.get("NIDS_FIG", CODE / "docs" / "figures" / "v2")); FIG.mkdir(parents=True, exist_ok=True)
ARGS = common.load_json(RUN / "config.json")["args"]   # model and gate hyper-parameters of the run
torch.set_grad_enabled(False)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

print("code commit :", COMMIT)
print("torch       :", torch.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)
print("notebook 04 :", SRC)
print("FU2 (06)    :", FU2)
print("figures to  :", FIG)

**Chart style.** One palette for every chart, the same as the dashboard: clean `#5b6675`, gate only `#c0392b`, soft
repair `#d68910`, relabel repair `#1e8449`, info/data `#2471a3`. Each saved chart also gets a CSV of exactly the numbers
it plots. **Look at:** nothing to check here; this cell only defines helpers.

In [ ]:
PAL = {"clean": "#5b6675", "poisoned": "#c0392b", "poisoned_repair": "#d68910", "poisoned_relabel": "#1e8449",
       "info": "#2471a3", "brand": "#6e2617"}
NAME = {"clean": "Clean", "poisoned": "Gate only", "poisoned_repair": "Soft repair", "poisoned_relabel": "Relabel repair"}
INK, INK2, GRID = "#1c1d1f", "#5f6368", "#e6e8eb"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "font.family": "DejaVu Sans",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "legend.frameon": False,
    "axes.titlelocation": "left", "axes.titlesize": 11.5, "axes.titlecolor": INK, "axes.titleweight": "bold",
    "text.color": INK, "figure.facecolor": "white", "axes.facecolor": "white"})
CLASSES4 = ["BENIGN", "Syn", "LDAP", "UDP"]


def save(fig, name, data: pd.DataFrame):
    png, csv = FIG / f"{name}.png", FIG / f"{name}_data.csv"
    fig.savefig(png, dpi=200, bbox_inches="tight", facecolor="white")
    data.to_csv(csv, index=False)
    print(f"saved {png.relative_to(CODE) if png.is_relative_to(CODE) else png} and {csv.name}")


def sha256(p):
    return common.sha256_file(Path(p))

## 2 · The raw data audit

**What it does:** reads the committed Stage 1 manifest (`experiments/results/01_data/split_manifest.json`) and tabulates,
per raw label: rows read, rows with non-finite values (dropped), exact duplicate rows (finite rows minus distinct feature
vectors), label-conflict vectors (one feature vector under two labels, all dropped) and rows kept after the per-attack
cap of 60,000. **Look at:** BENIGN is about 0.16% of the raw rows, and most attack rows are exact duplicates. That is why
all distinct BENIGN rows are kept and attack classes are capped.

In [ ]:
SM = common.load_json(RES / "01_data/split_manifest.json")
st = SM["stats"]
labels = sorted(st["raw_rows_by_label"], key=lambda k: -st["raw_rows_by_label"][k])
g = lambda key, k: st[key].get(k, 0)
audit = pd.DataFrame({
    "raw rows": [g("raw_rows_by_label", k) for k in labels],
    "non-finite (dropped)": [g("nonfinite_rows_by_label", k) for k in labels],
    "exact duplicates": [g("raw_rows_by_label", k) - g("nonfinite_rows_by_label", k) - g("distinct_rows_by_label", k) for k in labels],
    "distinct": [g("distinct_rows_by_label", k) for k in labels],
    "label-conflict vectors": [g("conflict_distinct_by_label", k) for k in labels],
    "kept": [g("kept_by_label", k) for k in labels]}, index=labels)
audit.loc["total"] = audit.sum()
audit["share of raw"] = audit["raw rows"] / audit.loc["total", "raw rows"]
display(audit.style.format({c: "{:,.0f}" for c in audit.columns if c != "share of raw"} | {"share of raw": "{:.2%}"}))
print(f"BENIGN share: {audit.loc['BENIGN', 'raw rows'] / audit.loc['total', 'raw rows']:.2%} of raw rows, "
      f"{audit.loc['BENIGN', 'kept'] / audit.loc['total', 'kept']:.1%} of kept rows | "
      f"exact duplicates: {audit.loc['total', 'exact duplicates'] / audit.loc['total', 'raw rows']:.1%} of raw rows | "
      f"files read: {len(st['rows_by_file'])}")

**What it does:** draws raw rows versus kept rows per label on a log scale. **Look at:** the gap between the two bars:
de-duplication and the cap shrink TFTP by more than 300×, while BENIGN loses only non-finite rows, duplicates and label conflicts.

In [ ]:
a = audit.drop(index="total").sort_values("raw rows")
fig, ax = plt.subplots(figsize=(8, 5.2))
y = np.arange(len(a))
ax.barh(y + 0.2, a["raw rows"], height=0.38, color=PAL["clean"], label="raw rows")
ax.barh(y - 0.2, a["kept"], height=0.38, color=PAL["info"], label="kept (distinct, capped)")
ax.set_xscale("log"); ax.set_yticks(y, a.index); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlabel("rows (log scale)"); ax.set_title("Raw vs kept rows per label")
ax.legend(loc="lower right"); plt.show()

## 3 · Cleaning and leakage control

**What it does:** lists every column the code drops before modelling, with the reason, straight from the constants in
`common.py` (`ID_COLS`, `PORT_COLS`, `LEAK_COLS`). **Look at:** identifiers, ports and the direction flag would let a
model recognise *who* sent a flow, or which service port it came from, instead of *how* it behaves.

In [ ]:
WHY = {
    "flow id": "identifier of the flow (5-tuple + time); memorises hosts, not behaviour",
    "source ip": "host identifier; attack and benign traffic come from different machines in the capture",
    "destination ip": "host identifier (the victim); same leak as source ip",
    "timestamp": "capture time; attacks were recorded in separate time windows, so time gives the label away",
    "unnamed: 0": "row index written by an earlier CSV export; position in the file, no traffic meaning",
    "unnamed: 0.1": "a second copy of that row index",
    "similarhttp": "metadata string present in some copies of the dataset, not a flow statistic",
    "simillarhttp": "the same metadata column under its misspelled raw name",
    "source port": "reflection attacks come from a fixed service port (e.g. 389 for LDAP): a near-perfect label giveaway",
    "destination port": "same giveaway on the other side of the flow",
    "inbound": "direction flag; possibly a label giveaway, dropped by default until audited",
}
dropped = sorted(common.ID_COLS) + sorted(common.PORT_COLS if common.DROP_PORTS else []) + sorted(common.LEAK_COLS if common.DROP_INBOUND else [])
assert set(dropped) == set(WHY), set(dropped) ^ set(WHY)
display(pd.DataFrame({"dropped column": dropped, "why": [WHY[c] for c in dropped]}))
print("also: rows with inf/NaN features dropped; exact duplicates removed globally; feature vectors seen under two labels removed")
print("model inputs:", len(common.load_json(RES / "01_data/features.json")["numeric"]), "numeric features + protocol (categorical)")

**What it does:** shows the stratified random 60/10/10/20 split for the four classes used in the main experiments,
the rows left after Stage 1b removed protected rows whose *scaled* input equals a training input, and the training rows
each experience actually uses (attack classes capped at 13,890 rows). **Look at:** the split fractions are exact, test
has 46,527 scored rows, and training never touches calib, val or test.

In [ ]:
cps = SM["counts_per_split"]
EM = common.load_json(RES / "01_data/cii/experience_manifest.json")
kept = EM["protected_exclusions"]["kept_by_split_class"]
split = pd.DataFrame({sp: [cps[sp][k] for k in CLASSES4] for sp in ("train", "calib", "val", "test")}, index=CLASSES4)
split["total"] = split.sum(axis=1)
for sp in ("train", "calib", "val", "test"):
    split[f"{sp} %"] = split[sp] / split["total"]
for sp in ("calib", "val", "test"):
    split[f"{sp} scored"] = [kept[sp][k] for k in CLASSES4]
display(split.style.format({c: "{:.1%}" for c in split.columns if c.endswith("%")} |
                           {c: "{:,}" for c in split.columns if not c.endswith("%")}))
exp = pd.DataFrame(EM["counts_per_experience"]).fillna(0).astype(int).reindex(CLASSES4).fillna(0).astype(int)
display(exp.rename(columns=lambda e: f"{e} train rows"))
print("schedule (CII):", " -> ".join("+".join(e) for e in EM["schedule"]), "| protected rows excluded:", EM["protected_exclusions"]["n_excluded"],
      "| scaled-input overlap after exclusion:", {k: v["equal_to_a_train_input"] for k, v in EM["scaled_input_overlap"].items()})

**What it does:** loads the frozen scaler (fitted once on E0 training rows only, then never changed) and shows its
mean and standard deviation, after the signed log, for five example features. It also checks that the scaler on Kaggle
is byte-identical to the committed copy. **Look at:** `fit_on`, and `True` for the identity check.

In [ ]:
SC = common.load_json(SRC / "cii/scaler.json")
same = sha256(SRC / "cii/scaler.json") == sha256(RES / "01_data/cii/scaler.json")
print("transform:", SC["transform"], "| fitted on:", SC["fit_on"], "| clip: ±", SC["clip"],
      "| constant in E0 (std floored):", len(SC["const_in_e0"]), "| Kaggle copy == committed copy:", same)
EX5 = ["flow duration", "total fwd packets", "fwd packet length max", "flow bytes/s", "packet length mean"]
display(S5.scaler_table(SC).set_index("feature").loc[EX5])

## 4 · One flow, end to end

**What it does:** takes the first LDAP row of the held-out test set (by row id, among the rows that are scored) and
shows its raw values for eight readable features. **Look at:** raw values span many orders of magnitude (bytes/s in the
millions next to packet counts of 2), which is why the scaler takes a signed log first.

In [ ]:
CM = common.load_json(SRC / "class_map.json")
EXCL = set(common.load_json(SRC / "protected_exclusions.json")["excluded_rids"])
TEST = pd.read_parquet(SRC / "clean_test.parquet")
ldap = TEST[(TEST[common.LABEL_COL] == "LDAP") & ~TEST["_rid"].isin(EXCL)].sort_values("_rid")
ROW = ldap.iloc[[0]]
FEATS8 = ["flow duration", "total fwd packets", "total length of fwd packets", "fwd packet length max",
          "flow bytes/s", "flow packets/s", "packet length mean", "init_win_bytes_forward"]
print(f"test row id {int(ROW['_rid'].iloc[0])} | true label {ROW[common.LABEL_COL].iloc[0]} | protocol {int(ROW['protocol'].iloc[0])}")
display(ROW[FEATS8].T.rename(columns=lambda _: "raw value"))

**What it does:** applies the frozen scaler step by step with `s5_explain.worked_example`: signed log
`sign(x)·log(1+|x|)`, then the z-score with the E0 mean and std, then clipping to ±5, then float32.
**Look at:** the `clip(z, ±5)` column; a value far outside the E0 range is capped at ±5 instead of dominating.

In [ ]:
display(S5.worked_example(ROW, SC, FEATS8).set_index("feature").round(4))

**What it does:** turns the scaled row into the model's input tokens: one learned 32-d token per numeric feature
(`x_j·W_j + b_j`), one embedding for the protocol, and a learned `[CLS]` token in front. **Look at:** the shape
`(1, 79, 32)` = 1 flow × (1 `[CLS]` + 77 features + 1 protocol) × 32 dimensions.

In [ ]:
X, C, _ = S1B.apply_scaler(ROW, SC, CM)
XT, CT = torch.from_numpy(X), torch.from_numpy(C)
RELABEL, REL_CLS = S5.model_from_ckpt(FU2RUN / "poisoned_relabel/A2.pt", ARGS)   # FU2: deployed relabel-repair model
GATE, GATE_CLS = S5.model_from_ckpt(RUN / "poisoned/A2.pt", ARGS)                # sweep: deployed gate-only model
tokens = RELABEL.tokenizer(XT, CT)
print("x_num:", tuple(XT.shape), "| x_cat (protocol id):", tuple(CT.shape), C.ravel().tolist(), "| tokens:", tuple(tokens.shape))

**What it does:** runs the two deployed models on this flow: the relabel-repair model (FU2, `fu2_relabel_f1.0_s42`,
`A2.pt`) and the gate-only model (`sweep_f1.0_s42/poisoned/A2.pt`). Both saw the same poisoned stream (every stored Syn
label flipped to BENIGN). **Look at:** relabel repair puts almost all probability on LDAP. The gate-only model never
accepted an update, so its head still has only BENIGN and Syn, and it calls this LDAP flow BENIGN.

In [ ]:
def proba(m, cls):
    p = torch.softmax(m(XT, CT), -1).numpy()[0]
    return {k: (float(p[cls.index(k)]) if k in cls else np.nan) for k in CLASSES4}

pv = pd.DataFrame({"relabel repair": proba(RELABEL, REL_CLS), "gate only": proba(GATE, GATE_CLS)}).T
display(pv.style.format(lambda v: "not in head" if pd.isna(v) else f"{v:.4f}"))
for name, row in pv.iterrows():
    print(f"{name:15s} predicts {row.idxmax()} (head classes: {REL_CLS if name == 'relabel repair' else GATE_CLS})")

**What it does:** captures the input of the last Transformer block with a forward pre-hook, recomputes that block's
self-attention with weights switched on, and plots how much the `[CLS]` token attends to each feature (averaged over
the 4 heads). **Look at:** the top-10 bars. Attention shows where the `[CLS]` token reads from, not a causal
explanation of the prediction.

In [ ]:
cap = {}
last = RELABEL.encoder.layers[-1]
hook = last.register_forward_pre_hook(lambda mod, inp: cap.__setitem__("h", inp[0]))
RELABEL(XT, CT)
hook.remove()
z = last.norm1(cap["h"]) if last.norm_first else cap["h"]
_, W = last.self_attn(z, z, z, need_weights=True, average_attn_weights=True)
att = W[0, 0].numpy()                                  # row of the [CLS] query: weights over the 79 tokens
names = ["[CLS]"] + SC["features_numeric"] + ["protocol"]
assert len(att) == len(names) and abs(att.sum() - 1) < 1e-5
top = pd.Series(att[1:], index=names[1:]).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh(top.index[::-1], top.values[::-1], color=PAL["info"], height=0.6)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_xlabel("attention weight from [CLS] (last layer, mean of 4 heads)")
ax.set_title("Where the [CLS] token looks for this LDAP flow (relabel model)")
plt.show()
print(f"[CLS] -> itself: {att[0]:.3f} | top-10 features hold {top.sum():.3f} of the attention")

## 5 · The model

**What it does:** prints the module tree of the deployed relabel model and counts its trainable parameters, also
grouped by role (`s5_explain.parameter_table`). **Look at:** 43,396 parameters in total: 3 pre-norm Transformer blocks
(d = 32, 4 heads), a per-feature tokenizer, and a 4-class head.

In [ ]:
print(RELABEL)
n_params = sum(p.numel() for p in RELABEL.parameters())
print(f"\ntrainable parameters: {n_params:,}")
assert n_params == 43_396
display(S5.parameter_table(RELABEL).groupby("role", sort=False)["n_params"].sum().to_frame())

**What it does:** demonstrates `TabTransformer.expand_head`, which adds output rows when a new class arrives. It loads
`A0` (BENIGN, Syn), copies it, expands the copy to 3 outputs, and checks that the old weight rows and the old logits are
unchanged. **Look at:** both checks print `True`; only the new row (random) is added, so old-class decisions survive
the expansion itself.

In [ ]:
A0, A0_CLS = S5.model_from_ckpt(RUN / "A0.pt", ARGS)
before_w, before_logits = A0.head.weight.clone(), A0(XT, CT)
M = copy.deepcopy(A0); M.expand_head(3)
after_logits = M(XT, CT)
print("classes:", A0_CLS, "| head weight", tuple(A0.head.weight.shape), "->", tuple(M.head.weight.shape))
print("old rows copied unchanged :", torch.equal(M.head.weight[:2], before_w) and torch.equal(M.head.bias[:2], A0.head.bias))
print("old logits unchanged      :", torch.allclose(after_logits[:, :2], before_logits, atol=1e-6),
      "| logits before", before_logits.numpy().round(3).tolist(), "after", after_logits.numpy().round(3).tolist())

## 6 · The poisoned buffer

**What it does:** loads the replay buffer the update at E1 trains on in run `sweep_f1.0_s42` (2,000 stored rows per old
class) and counts labels before flipping (true labels) and after flipping (the labels actually stored). **Look at:**
at f = 1.0, all 2,000 stored Syn rows now carry the label BENIGN, and no correct Syn label is left in memory.

In [ ]:
BUF = pd.read_csv(RUN / "poisoned_repair/buffer_before_E1.csv")
comp = pd.DataFrame({"before flipping (true label)": BUF["true_label"].value_counts(),
                     "after flipping (stored label)": BUF["label"].value_counts()}).fillna(0).astype(int)
display(comp)
display(pd.crosstab(BUF["true_label"], BUF["label"], rownames=["true"], colnames=["stored"]))
print(f"flipped rows: {int(BUF['flipped'].sum()):,} of {len(BUF):,} | poison fraction in config: {ARGS['poison_frac']}")

**What it does:** reads the frozen teacher's view of that buffer from `rounds.json`. The soft-repair weight of a
replay row is the teacher's probability of its *stored* label, `p_T(stored | x)`. **Look at:** flipped rows get a mean
weight of almost 0 (the teacher is sure they are Syn, not BENIGN); correctly labelled rows get almost 1. At f = 1.0 that
is not enough: no correct Syn row is left, and soft repair is still rejected or forgets Syn. The relabel repair (FU2)
instead relabels the doubted rows to the teacher's class.

In [ ]:
rounds = common.load_json(RUN / "poisoned_repair/rounds.json")
tw = pd.DataFrame([{"experience": f"E{r['experience']}", "attempt": r["attempt"], "accepted": r["accepted"],
                    "replay rows": r["n_replay"], "flipped": r["n_replay_flipped"],
                    "mean p_T(stored) on flipped": r["teacher_weights"]["mean_w_flipped"],
                    "mean p_T(stored) on correct rows": r["teacher_weights"]["mean_w_clean_known"]}
                   for r in rounds if r.get("teacher_weights")])
display(tw.round(4))
fs = common.load_json(RES / "06_fu2/fu2/cii/fu2_relabel_f1.0_s42/summary.json")
display(pd.DataFrame(fs["relabel_stats"]).set_index("experience")[["n_relabelled", "n_relabelled_to_true", "n_quarantined"]]
        .rename(columns=lambda c: "FU2 " + c))

## 7 · The gate decision, recomputed

**What it does:** takes FU2 seed 42 at E1 (C1 = plain update, R1 = relabel repair). It recomputes every gate inequality
from the validation metrics saved in the committed `rounds.json` with `s5_explain.gate_check`, which also asserts that
the recomputed decision equals the saved one. **Look at:** C1 fails only the old-Syn floor (Syn recall collapsed). R1
passes every check, and the `margin` column shows by how much.

In [ ]:
F_ROUNDS = RES / "06_fu2/fu2/cii/fu2_relabel_f1.0_s42/poisoned_relabel/rounds.json"
print("committed rounds.json == Kaggle FU2 output:", sha256(F_ROUNDS) == sha256(FU2RUN / "poisoned_relabel/rounds.json"))
fr = [r for r in common.load_json(F_ROUNDS) if r["experience"] == 1]
gc = S5.gate_check(fr, ARGS)
display(gc.style.format({"value": "{:.4f}", "bound": "{:.4f}", "margin": "{:+.4f}"})
          .map(lambda v: f"color: {PAL['poisoned_relabel'] if v else PAL['poisoned']}; font-weight: bold", subset=["pass"]))
for r in fr:
    print(f"E1 {r['attempt']}: saved decision {'ACCEPT' if r['accepted'] else 'REJECT'}"
          + (f" ({'; '.join(r['gate_reasons'])})" if r["gate_reasons"] else "") + " = recomputed")

## 8 · Results from committed files

The six headline charts, read through the dashboard's own loaders (`dashboard/build_dashboard.py`, committed files only),
so the notebook, the dashboard and `docs/RESULT_TABLES.md` show the same numbers. Each chart is saved as a 200-dpi PNG in
`docs/figures/v2/` with a CSV of the plotted numbers next to it. **Look at:** nothing to check here; this cell loads the
data.

In [ ]:
D = BD.results()
FUP = {f["id"]: f for f in D["followups"]}
print("loaded:", ", ".join(k for k in D if k not in ("followups", "canon")), "| follow-ups:", ", ".join(FUP))

### a) Seeds that learn all classes, per setting and defence (pooled seeds 42–49)

**Takeaway:** over seeds 42–49, soft repair learns all four classes in 24/24 runs at f 0.75–0.95 (gate only 8/24), and
relabel repair in 8/8 at f 1.0 (gate only 0/8). The five-class settings repeat the pattern (8/8 vs 0/8).

In [ ]:
sd = FUP["seeds"]
rows = [{"setting": g["label"], "arm": b["arm"], "learned_all": b["k"], "runs": b["n"], "share": b["k"] / b["n"]}
        for g in sd["chart"] for b in g["bars"]]
dfa = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 4.2))
sets = list(dict.fromkeys(dfa["setting"])); w = 0.26
for gi, s in enumerate(sets):
    sub = dfa[dfa["setting"] == s].reset_index(drop=True)
    for bi, r in sub.iterrows():
        x = gi + (bi - 1) * (w + 0.02)
        ax.bar(x, r["share"], width=w, color=PAL[r["arm"]])
        ax.text(x, r["share"] + 0.02, f"{r['learned_all']}/{r['runs']}", ha="center", fontsize=8.5, color=INK)
ax.set_xticks(range(len(sets)), sets); ax.set_ylim(0, 1.12); ax.set_yticks([0, .5, 1], ["0%", "50%", "100%"])
ax.set_ylabel("runs learning every class"); ax.set_title(f"a · All classes learned, pooled seeds {sd['pooled']['seeds']}")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PAL[k]) for k in ("poisoned", "poisoned_repair", "poisoned_relabel", "clean")],
          labels=["Gate only", "Soft repair", "Relabel repair", "Clean"], ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
save(fig, "fig_a_all_classes_learned", dfa); plt.show()
print("takeaway (computed):", sd["takeaway"])

### b) Per-class recall and precision of the f = 1.0, seed-42 models

**What it does:** scores the four deployed models on the full held-out test set with `s3_continual.evaluate` (inference
only, CPU), checks every recall against the committed test results, and computes precision from the confusion matrices.
**Takeaway:** relabel repair's recall is within 0.0013 of the clean model on every class; gate only and soft repair never
predict LDAP or UDP, and their BENIGN precision falls to 0.55 because every LDAP and UDP flow is called BENIGN.

In [ ]:
from argparse import Namespace
t0 = time.time()
CTX, *_ = S3.load_context(SRC, "", "cii", "cpu")
PATHS = {"clean": RUN / "clean/A2.pt", "poisoned": RUN / "poisoned/A2.pt", "poisoned_repair": RUN / "poisoned_repair/A2.pt",
         "poisoned_relabel": FU2RUN / "poisoned_relabel/A2.pt"}
EV = {}
for arm, p in PATHS.items():
    m, cls = S3.load_ckpt(p, CTX, Namespace(**ARGS))
    EV[arm] = S3.evaluate(m, cls, CTX["test"], CTX["device"], scope=CTX["scope"]) | {"classes": cls}
FT = pd.read_csv(RES / "04_sweep/final_test.csv").query("run == 'sweep_f1.0_s42'").set_index("arm")
FU2T = common.load_json(RES / "06_fu2/test/fu2/cii/fu2_relabel_f1.0_s42/final_test.json")["test"]["recall"]
for arm, e in EV.items():
    want = FU2T if arm == "poisoned_relabel" else {k: FT.loc[arm, f"recall_{k}"] for k in CLASSES4}
    diff = max(abs(e["recall"][k] - float(want[k])) for k in CLASSES4)
    assert diff < 1e-9, (arm, diff)
    print(f"{NAME[arm]:15s} rows {e['n_rows']:,} | recall == committed test recall (max diff {diff:.1e})")
print(f"scored in {time.time() - t0:.0f}s on CPU")

def precision(conf, c):
    col = sum(conf[t].get(c, 0) for t in conf)
    return conf[c][c] / col if col and c in conf and c in conf[c] else np.nan

dfb = pd.DataFrame([{"arm": a, "class": c, "recall": EV[a]["recall"][c],
                     "precision": precision(EV[a]["confusion"], c) if c in EV[a]["classes"] else np.nan,
                     "class_learned": c in EV[a]["classes"]} for a in PATHS for c in CLASSES4])
fig, axes = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True)
w = 0.2
for ax, kind in zip(axes, ("recall", "precision")):
    for ai, a in enumerate(PATHS):
        sub = dfb[dfb["arm"] == a]
        xs = np.arange(4) + (ai - 1.5) * (w + 0.01)
        ax.bar(xs, sub[kind].fillna(0), width=w, color=PAL[a], label=NAME[a])
        for x, ok in zip(xs, sub["class_learned"]):
            if not ok:
                ax.text(x, 0.03, "×", ha="center", fontsize=10, color=INK2)
    ax.set_ylim(0, 1.05); ax.set_ylabel(kind)
axes[1].set_xticks(range(4), CLASSES4)
axes[0].set_title("b · Per-class recall and precision, f = 1.0, seed 42 (held-out test, 46,527 rows)")
axes[1].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), title="× = class not in the model's head")
save(fig, "fig_b_per_class_recall_precision", dfb); plt.show()
d_rel = max(abs(EV["poisoned_relabel"]["recall"][c] - EV["clean"]["recall"][c]) for c in CLASSES4)
print(f"takeaway (computed): relabel repair within {d_rel:.4f} of clean on every class; never predicting LDAP/UDP:",
      [NAME[a] for a in PATHS if "LDAP" not in EV[a]["classes"]])

### c) Accuracy versus macro recall (why accuracy misleads)

**Takeaway:** gate only still scores 0.664 accuracy, but its macro recall is 0.499: two attack classes are never
detected. Accuracy is carried by the large BENIGN and Syn classes; macro recall weighs every class equally.

In [ ]:
dfc = pd.DataFrame([{"arm": a, "accuracy": EV[a]["accuracy"], "macro_recall": EV[a]["macro_recall"]} for a in PATHS])
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, r in dfc.iterrows():
    ax.bar(i - 0.18, r["accuracy"], width=0.34, color=PAL[r["arm"]])
    ax.bar(i + 0.18, r["macro_recall"], width=0.34, color=PAL[r["arm"]], alpha=0.45)
    ax.text(i - 0.18, r["accuracy"] + 0.015, f"{r['accuracy']:.3f}", ha="center", fontsize=8.5)
    ax.text(i + 0.18, r["macro_recall"] + 0.015, f"{r['macro_recall']:.3f}", ha="center", fontsize=8.5)
ax.set_xticks(range(len(dfc)), [NAME[a] for a in dfc["arm"]]); ax.set_ylim(0, 1.1)
ax.set_title("c · Accuracy (solid) vs macro recall (light), f = 1.0, seed 42")
save(fig, "fig_c_accuracy_vs_macro_recall", dfc); plt.show()
g = dfc.set_index("arm").loc["poisoned"]
print(f"takeaway (computed): gate only accuracy {g['accuracy']:.3f}, macro recall {g['macro_recall']:.3f}")

### d) The RF confidence trajectory

**What it does:** plots the refresh experiment (RF, post hoc): before each of six updates, another 10% of the stored
Syn labels are flipped (validation, mean of 3 seeds). **Takeaway:** Syn recall stays at 0.994 while the model's mean
confidence on Syn falls to 0.770 (clean 0.992). A recall-only gate never fires.

In [ ]:
tr = pd.DataFrame(D["rf_traj"]).query("abs(rho - 0.1) < 1e-9")
m_ = tr.groupby(["arm", "t"])[["Syn_recall", "Syn_mean_p"]].mean().reset_index()
P_, C_ = m_[m_["arm"] == "poisoned"], m_[m_["arm"] == "clean"]
dfd = pd.DataFrame({"t": P_["t"].astype(int).values, "poisoned_Syn_recall": P_["Syn_recall"].values,
                    "poisoned_Syn_confidence": P_["Syn_mean_p"].values, "clean_Syn_confidence": C_["Syn_mean_p"].values})
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(dfd["t"], dfd["poisoned_Syn_recall"], color=PAL["poisoned"], lw=2, marker="o", ms=4, label="poisoned: Syn recall")
ax.plot(dfd["t"], dfd["poisoned_Syn_confidence"], color=PAL["poisoned"], lw=2, ls="--", marker="o", ms=4, label="poisoned: Syn confidence")
ax.plot(dfd["t"], dfd["clean_Syn_confidence"], color=PAL["clean"], lw=2, ls=":", marker="o", ms=4, label="clean: Syn confidence")
for col, dy in (("poisoned_Syn_recall", 7), ("poisoned_Syn_confidence", 0), ("clean_Syn_confidence", -8)):
    ax.annotate(f"{dfd[col].iloc[-1]:.3f}", (6, dfd[col].iloc[-1]), xytext=(7, dy), textcoords="offset points", va="center", fontsize=8.5)
ax.set_xticks(range(7), ["A0"] + [f"U{t}" for t in range(1, 7)]); ax.set_ylim(0.7, 1.02); ax.set_xlim(-0.2, 6.6)
ax.set_title("d · RF: recall holds while confidence falls (validation, 3 seeds)"); ax.legend(loc="lower left")
save(fig, "fig_d_rf_confidence_trajectory", dfd); plt.show()
print(f"takeaway (computed): after U6 Syn recall {dfd['poisoned_Syn_recall'].iloc[-1]:.3f}, confidence "
      f"{dfd['poisoned_Syn_confidence'].iloc[-1]:.3f} (clean {dfd['clean_Syn_confidence'].iloc[-1]:.3f})")

### e) The confidence-gate follow-up (CG, post hoc)

**What it does:** compares the mean held-out Syn confidence with and without a gate check on confidence (reject an update
if an old attack class's mean validation confidence drops by more than a fixed margin). **Takeaway:** with the
confidence gate, repair keeps Syn confidence at 0.992 (without it: 0.771), and 42/42 clean updates still pass.

In [ ]:
cg = pd.DataFrame(FUP["confgate"]["stats"])
rft = pd.DataFrame(D["rf_test"])
SET = [("CG-RF", 0.1, "Refresh, 10%/update"), ("CG-RF", 0.05, "Refresh, 5%/update"), ("CG-MAIN", 0.5, "4 classes, 50% flipped")]
rows = []
for st_, f, lab in SET:
    s = cg[(cg["setting"] == st_) & ((cg["frac"] - f).abs() < 1e-9)].set_index("arm")["Syn_mean_conf"]
    no = rft[((rft["rho"] - f).abs() < 1e-9) & (rft["arm"] == "poisoned_repair")]["Syn_mean_p"].mean() if st_ == "CG-RF" else np.nan
    rows.append({"setting": lab, "repair_no_conf_gate": no, "repair_with_conf_gate": s["poisoned_repair"],
                 "gate_only": s["poisoned"], "clean": s["clean"]})
dfe = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 3.6))
mk = {"repair_no_conf_gate": ("o", "white", PAL["poisoned_repair"], "repair, no confidence gate (RF)", -0.21),
      "repair_with_conf_gate": ("o", PAL["poisoned_repair"], "white", "repair + confidence gate", -0.07),
      "gate_only": ("D", PAL["poisoned"], "white", "gate only", 0.07), "clean": ("s", PAL["clean"], "white", "clean", 0.21)}
for i, r in dfe.iterrows():
    for k, (m, fc, ec, lab, dy) in mk.items():
        if pd.notna(r[k]):
            ax.scatter(r[k], i + dy, marker=m, s=70, facecolors=fc, edgecolors=ec, linewidths=1.8 if fc == "white" else 1.2, label=lab, zorder=3)
ax.set_yticks(range(len(dfe)), dfe["setting"]); ax.invert_yaxis(); ax.set_xlim(0.7, 1.005); ax.set_ylim(len(dfe) - 0.55, -0.45)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_xlabel("mean Syn confidence on held-out test (3 seeds)")
h, l = ax.get_legend_handles_labels(); u = dict(zip(l, h))
ax.legend(u.values(), u.keys(), ncol=2, loc="upper center", bbox_to_anchor=(0.45, -0.25))
ax.set_title("e · Confidence-aware gate: Syn confidence on test")
save(fig, "fig_e_confidence_gate", dfe); plt.show()
cl = cg[cg["arm"] == "clean"]
w10 = dfe.iloc[0]
print(f"takeaway (computed): refresh 10%: with {w10['repair_with_conf_gate']:.3f} vs without {w10['repair_no_conf_gate']:.3f}; "
      f"clean updates accepted {int(cl['updates_accepted'].sum())}/{int(cl['updates'].sum())}")

### f) The five-class follow-up (FIVE, post hoc)

**What it does:** adds a fourth update that brings TFTP (BENIGN + Syn, then + LDAP, + UDP, + TFTP), seeds 42–44, held-out
test. **Takeaway:** relabel repair still learns all five classes in 3/3 seeds at 100% flipped (gate only 0/3), soft repair
3/3 at 90%, and every repaired model keeps benign FAR ≤ 0.0020.

In [ ]:
fv = pd.DataFrame(FUP["five"]["stats"])
dff = fv.assign(defence=lambda d: np.where(d["arm"] == "poisoned_repair", np.where((d["frac"] - 1.0).abs() < 1e-9, "poisoned_relabel", "poisoned_repair"), d["arm"]),
                share=lambda d: d["all5"] / d["n_test"], max_benign_far=lambda d: d["benign_far"].map(max))[
    ["frac", "arm", "defence", "all5", "n_test", "share", "max_benign_far"]]
fig, ax = plt.subplots(figsize=(8, 3.8))
for gi, f in enumerate((0.9, 1.0)):
    sub = dff[(dff["frac"] - f).abs() < 1e-9].set_index("arm").loc[["poisoned", "poisoned_repair", "clean"]]
    for bi, (arm, r) in enumerate(sub.iterrows()):
        x = gi + (bi - 1) * 0.28
        ax.bar(x, r["share"], width=0.26, color=PAL[r["defence"]])
        ax.text(x, r["share"] + 0.02, f"{r['all5']}/{r['n_test']}", ha="center", fontsize=8.5)
ax.set_xticks([0, 1], ["90% flipped (soft repair)", "100% flipped (relabel repair)"]); ax.set_ylim(0, 1.12)
ax.set_yticks([0, 1/3, 2/3, 1], ["0/3", "1/3", "2/3", "3/3"]); ax.set_ylabel("seeds learning all 5 classes")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=PAL[k]) for k in ("poisoned", "poisoned_repair", "poisoned_relabel", "clean")],
          labels=["Gate only", "Soft repair (90%)", "Relabel repair (100%)", "Clean"], ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("f · A fifth class (TFTP): all five learned (held-out test, seeds 42–44)")
save(fig, "fig_f_five_class", dff); plt.show()
rep = dff[dff["arm"] == "poisoned_repair"]
print(f"takeaway (computed): repaired models' max benign FAR {rep['max_benign_far'].max():.4f}")

## 9 · Integrity

**What it does:** runs `make_report_tables.py --check`, which rebuilds `docs/RESULT_TABLES.md` from the committed files
and fails on any difference. **Look at:** the `OK` line and return code 0.

In [ ]:
p = subprocess.run([sys.executable, "make_report_tables.py", "--check"], cwd=CODE, capture_output=True, text=True)
print(p.stdout.strip()[-1500:], p.stderr.strip()[-1500:], sep="\n")
print("return code:", p.returncode)
assert p.returncode == 0

**What it does:** checks the frozen teacher and the deployed models by sha256. The teacher's weights hash
(`s3_continual.state_hash`, the value s3 writes as `T_sha256`) must equal the hash in its run manifest and in the
committed FU2 summary, and every committed run manifest must report
the teacher unchanged at the end of its run. **Look at:** every line ends in `True`.

In [ ]:
rm = common.load_json(RUN / "run_manifest.json")
T_MODEL, T_CLS = S3.load_ckpt(RUN / "T.pt", CTX, Namespace(**ARGS))
t_sha = S3.state_hash(T_MODEL)   # sha256 over the weight tensors, exactly as s3 records T_sha256
print("teacher T.pt", T_CLS, "| file sha256:", sha256(RUN / "T.pt")[:16] + "…")
print("teacher weights sha256    :", t_sha)
print("  == run manifest          :", t_sha == rm["T_sha256"], "| T unchanged at end of run:", rm["T_unchanged"])
print("  == FU2 summary (committed):", t_sha == fs["T_sha256"], "| FU2 T unchanged:", fs["T_unchanged"])
print("FU2 A0 == committed sha    :", sha256(FU2RUN / "A0.pt") == fs["A0_sha256"])
print("FU2 A2 == committed sha    :", sha256(FU2RUN / "poisoned_relabel/A2.pt") == fs["final_sha256"])
assets = common.load_json(CODE / "dashboard/dashboard_assets.json")["meta"]["model_sha256"]
print("dashboard models == these  :", all(sha256(PATHS[a]) == assets[a] for a in PATHS))
mans = sorted(RES.glob("1[1-3]_*/runs/cii/*/run_manifest.json")) + sorted(RES.glob("10_rf/runs/cii/*/run_manifest.json"))
unchanged = [common.load_json(m)["T_unchanged"] for m in mans]
print(f"committed run manifests reporting T unchanged: {sum(unchanged)}/{len(unchanged)} (RF, CG, FIVE, SEEDS):", all(unchanged))
assert t_sha == rm["T_sha256"] == fs["T_sha256"] and all(unchanged)

**What it does:** collects the test-row fingerprint (`rows_sha256`, a hash of exactly which rows were scored) from every
committed test result and from the four models scored above. **Look at:** one fingerprint for all four-class results
and one for all five-class results, so every arm, seed and follow-up is compared on identical rows.

In [ ]:
fp = []
fp += [("this notebook", a, e["rows_sha256"], 4) for a, e in EV.items()]
fp += [("04_sweep final_test.csv", r.run + "/" + r.arm, r.test_rows_sha256, 4) for r in pd.read_csv(RES / "04_sweep/final_test.csv").itertuples()]
fp += [("06_fu2 test_recall.csv", f"s{r.seed}/{r.arm}", r.test_rows_sha256, 4) for r in pd.read_csv(RES / "06_fu2/test/test_recall.csv").itertuples()]
for name, key in (("12_five test.json", "12_five/test.json"), ("13_seeds test.json", "13_seeds/test.json")):
    for r in common.load_json(RES / key)["runs"]:
        for a in ("clean", "poisoned", "poisoned_repair"):
            fp.append((name, f"{r['name']}/{a}", r[a]["rows_sha256"], r.get("classes", 5)))
fp = pd.DataFrame(fp, columns=["source", "model", "rows_sha256", "classes"])
summary = fp.groupby(["classes", "rows_sha256"]).agg(models=("model", "size"), sources=("source", lambda s: ", ".join(dict.fromkeys(s)))).reset_index()
summary["rows_sha256"] = summary["rows_sha256"].str[:16] + "…"
display(summary)
print("one fingerprint per schedule:", fp.groupby("classes")["rows_sha256"].nunique().to_dict())
assert (fp.groupby("classes")["rows_sha256"].nunique() == 1).all()
print("figure files:", sorted(p.name for p in FIG.iterdir()))

**Done.** Everything above was computed from the committed files and the saved checkpoints by the repository's own code;
nothing was trained and no result file was changed. The six figures and their data are in `docs/figures/v2/`.

**Kaggle only: save an executed copy.** Kaggle keeps the executed notebook internally, and its command-line tool returns
only the source. This cell finds this notebook's source on disk, re-runs every cell above it in a fresh kernel (same inputs,
same code), and writes the result with outputs to `/kaggle/working/12_walkthrough_end_to_end.ipynb`. That file is the one
committed to the repository, and it leaves out this cell. **Look at:** `errors: 0`.

In [ ]:
import nbformat
from nbclient import NotebookClient
MARK = "12 · End-to-end walkthrough"
OUT = Path(os.environ.get("NIDS_NBOUT", "/kaggle/working/12_walkthrough_end_to_end.ipynb"))
cands = []
for root in dict.fromkeys([START_CWD, "/kaggle/src", "/kaggle/working", "/kaggle", "/tmp", os.path.expanduser("~")]):
    cands += glob.glob(os.path.join(root, "*.ipynb")) + glob.glob(os.path.join(root, "*", "*.ipynb"))
src = next((q for q in cands if Path(q).resolve() != OUT.resolve() and MARK in Path(q).read_text(errors="ignore")), None)
if src is None:
    print("source notebook not found; searched", len(cands), "files under", START_CWD, "and /kaggle (not on Kaggle?)")
else:
    nb = nbformat.read(src, as_version=4)
    nb.cells = [c for c in nb.cells if "save-executed-copy" not in c.get("metadata", {}).get("tags", [])]
    t0 = time.time()
    NotebookClient(nb, timeout=1800, kernel_name="python3", allow_errors=True,
                   resources={"metadata": {"path": START_CWD}}).execute()
    errs = [o.get("ename") for c in nb.cells if c.cell_type == "code" for o in c.get("outputs", []) if o.get("output_type") == "error"]
    nbformat.write(nb, OUT)
    print(f"source: {src} | re-executed {sum(c.cell_type == 'code' for c in nb.cells)} code cells in {time.time() - t0:.0f}s"
          f" | errors: {len(errs)} | wrote {OUT} ({OUT.stat().st_size / 1e6:.2f} MB)")
    assert not errs, errs